In [ ]:
# The card's runtime stack is torch 2.9.1+cu128 (checked in the next cell) with
# transformers 5.0.0. torchcodec is part of that stack only for torchaudio.load,
# which the worker does not use - it reads references with soundfile and hands
# them to the processor's own encoder - so it is not installed.
!pip -q install "transformers==5.0.0" "accelerate>=1.10.1" speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime 2>&1 | tail -5


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch

# An earlier kernel's transformers install pulled a torchvision built for
# another torch, and transformers' lazy import of PreTrainedModel died on it.
# If that happens here, torchvision goes - nothing in this run loads a vision
# model - and the removal is printed, not hidden.
probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                       text=True, capture_output=True)
if probe.returncode != 0 and "torchvision" in probe.stderr:
    print("transformers cannot import beside this torchvision - removing torchvision")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"])
    probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                           text=True, capture_output=True)
assert probe.returncode == 0, probe.stderr[-2000:]

import transformers

print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
assert torch.__version__.startswith("2.9.1"), "the official stack pins torch 2.9.1+cu128"
assert transformers.__version__ == "5.0.0", "the official stack pins transformers 5.0.0"
gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("gpus        :", gpus)
print("capability  :", [torch.cuda.get_device_capability(i) for i in range(len(gpus))])
for i in range(len(gpus)):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}      : {free / 2**30:.1f} GiB free of {total / 2**30:.1f}")
print("disk        :", f"{shutil.disk_usage('/root').free / 2**30:.0f} GiB free for the HF cache")
# The model (9.1 GB bf16) and the codec (about 4.3 GB bf16) together, plus the
# KV cache, do not fit one 15 GiB T4 with room to spare. The second card holds
# the codec. That is placement; precision is untouched.
assert len(gpus) >= 2, "this recipe places the codec on a second GPU - use T4 x2"

INPUT = Path("/kaggle/input")
found = sorted(INPUT.rglob("repo/scripts/benchmark.py"))
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo under /kaggle/input - see the directories above")
REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(found[0].parent.parent, REPO)
assert (REPO / "scripts/moss15_worker.py").is_file(), "the dataset predates the MOSS v1.5 worker"

os.environ["VOICECLONE_MOSS15_PYTHON"] = sys.executable
os.environ["PYTHONPATH"] = str(REPO / "src")
os.environ["VOICECLONE_SCORING_ENV"] = "kaggle:persian-voice-moss15-v1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Sequential weight materialisation. The v1.0 run hung twice inside
# transformers' threaded loader and finished once this was set. It changes how
# weights are loaded, not what is computed.
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"


def run(command, timeout=None, cwd=None):
    print("$", " ".join(str(part) for part in command), flush=True)
    start = time.time()
    try:
        code = subprocess.run(command, cwd=cwd, text=True, timeout=timeout).returncode
    except subprocess.TimeoutExpired:
        print(f"TIMED OUT after {timeout}s - called off", flush=True)
        code = -1
    elapsed = time.time() - start
    print(f"exit {code} in {elapsed / 60:.1f} min", flush=True)
    return code, elapsed


In [ ]:
from huggingface_hub import snapshot_download

start = time.time()
MODEL_DIR = Path(snapshot_download(repo_id="OpenMOSS-Team/MOSS-TTS-Local-Transformer-v1.5", revision="be7766a6735b98bd793f7c79fb720b4d0f5d13b8"))
print(f"model in {(time.time() - start) / 60:.1f} min -> {MODEL_DIR}")
start = time.time()
CODEC_DIR = Path(snapshot_download(repo_id="OpenMOSS-Team/MOSS-Audio-Tokenizer-v2", revision="f6e20e543b33d2c252a7ef71bdf8aa71e5ff9169"))
print(f"codec in {(time.time() - start) / 60:.1f} min -> {CODEC_DIR}")
for folder in (MODEL_DIR, CODEC_DIR):
    size = sum(p.stat().st_size for p in folder.rglob("*") if p.is_file())
    print(f"   {size / 2**30:5.2f} GiB  {folder.name}")


In [ ]:
driver = Path("/kaggle/working/drive_moss15.py")
driver.write_text("""
import json, subprocess, sys, time

model_dir, codec_dir, reference, text, output, cwd = sys.argv[1:7]
command = [sys.executable, "scripts/moss15_worker.py", "--model-dir", model_dir,
           "--codec-dir", codec_dir, "--device", "cuda", "--codec-device", "auto"]
start = time.time()
worker = subprocess.Popen(command, cwd=cwd, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                          text=True, bufsize=1)

def reply():
    while True:
        line = worker.stdout.readline()
        if not line:
            raise SystemExit(f"worker exited with {worker.poll()}")
        try:
            return json.loads(line.strip())
        except json.JSONDecodeError:
            continue

ready = reply()
print("READY", json.dumps(ready), flush=True)
print(f"LOAD {time.time() - start:.1f}s", flush=True)
if not ready.get("ready"):
    raise SystemExit("worker did not load")
job = {"text": text, "reference": reference, "output": output, "seed": 42, "language": "Persian",
       "audio_temperature": 1.7, "audio_top_p": 0.8, "audio_top_k": 25,
       "audio_repetition_penalty": 1.0, "max_new_tokens": 4096}
start = time.time()
worker.stdin.write(json.dumps(job, ensure_ascii=False) + chr(10))
worker.stdin.flush()
answer = reply()
print(f"RENDER {time.time() - start:.1f}s", flush=True)
print("RESULT", json.dumps(answer), flush=True)
worker.stdin.write(json.dumps({"op": "quit"}) + chr(10))
worker.stdin.flush()
""", encoding="utf-8")

manifest = json.loads((REPO / "eval/corpus/manifest.json").read_text(encoding="utf-8"))
entry = next(e for e in manifest["entries"] if e["speaker"] == "cv02" and e["role"] == "reference")
REFERENCE = REPO / "eval/corpus" / entry["file"]
SENTENCE = "دیروز در کتابخانه شهر، کتابی درباره تاریخ ایران خواندم."
RENDER = Path("/kaggle/working/moss15_worker.wav")


In [ ]:
worker_code, worker_seconds = run(
    [sys.executable, "/kaggle/working/drive_moss15.py", str(MODEL_DIR), str(CODEC_DIR),
     str(REFERENCE), SENTENCE, str(RENDER), str(REPO)],
    timeout=3600,
)
subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv"])

if RENDER.is_file():
    import numpy as np
    import soundfile as sf

    samples, rate = sf.read(RENDER, always_2d=True)
    flat = np.asarray(samples, dtype=np.float64).reshape(-1)
    print()
    print(f"render      : {samples.shape[0] / rate:.2f} s, {rate} Hz, {samples.shape[1]} channel(s)")
    print(f"peak / rms  : {float(np.abs(flat).max()):.3f} / {float(np.sqrt((flat ** 2).mean())):.4f}")
    print(f"distinct    : {len(np.unique(flat))} sample values")
    verdict = []
    if rate != 48000 or samples.shape[1] != 2:
        verdict.append("NOT THE NATIVE 48 kHz STEREO THE CARD DESCRIBES")
    if not np.isfinite(flat).all():
        verdict.append("NOT FINITE")
    if len(np.unique(flat)) < 1000:
        verdict.append("CONSTANT OR NEARLY SO")
    print("VERDICT     :", " | ".join(verdict) if verdict else "native-format audio")
else:
    print("no render was written - the log above says why; nothing later will run the model")


In [ ]:
code, elapsed = run([sys.executable, "scripts/benchmark.py", "--label", "moss15-smoke",
                     "--engine", "moss15", "--device", "cuda", "--speakers", "cv02", "--sentences", "s1"],
                    timeout=3600, cwd=str(REPO))
path = REPO / "eval/results/moss15-smoke.json"
if path.is_file():
    data = json.loads(path.read_text())
    print("summary:", json.dumps(data["summary"], ensure_ascii=False))
    print("environment:", json.dumps(data.get("scoring_environment"), ensure_ascii=False))
else:
    print("no results file")


In [ ]:
code, elapsed = run([sys.executable, "scripts/benchmark.py", "--label", "baseline-moss15-env",
                     "--score-only", "--replay-from", "baseline", "--device", "cuda"],
                    timeout=3600, cwd=str(REPO))
sys.path.insert(0, str(REPO / "scripts"))
import compare_runs

reference = compare_runs.load("baseline", REPO / "eval/results")
control = compare_runs.load("baseline-moss15-env", REPO / "eval/results")
check = compare_runs.scale_control(reference, control)
print("scale control, cell by cell:", {k: (len(v) if isinstance(v, (list, dict)) else v)
                                       for k, v in check.items()})


In [ ]:
smoke = REPO / "eval/results/moss15-smoke.json"
ready = False
if smoke.is_file():
    data = json.loads(smoke.read_text())
    row = (data["rows"] or [{}])[0]
    ready = data["summary"]["renders"] == 1 and row.get("held_out_similarity") is not None
if not ready:
    print("no scored render from the smoke cell - not starting the development benchmark")
else:
    code, elapsed = run([sys.executable, "scripts/benchmark.py", "--label", "moss15",
                         "--engine", "moss15", "--device", "cuda"],
                        timeout=14400, cwd=str(REPO))
    path = REPO / "eval/results/moss15.json"
    print(json.dumps(json.loads(path.read_text())["summary"], indent=2) if path.is_file() else "no results")


In [ ]:
destination = Path("/kaggle/working/results")
destination.mkdir(exist_ok=True)
for path in (REPO / "eval/results").glob("*.json"):
    shutil.copy2(path, destination / path.name)
native = REPO / "outputs/native/moss15"
if native.is_dir():
    shutil.copytree(native, Path("/kaggle/working/moss15_native"), dirs_exist_ok=True)
renders = Path("/kaggle/working/moss15_renders")
renders.mkdir(exist_ok=True)
for folder in ("moss15-smoke", "moss15"):
    for path in sorted((REPO / "outputs/benchmark" / folder).glob("*.wav")):
        shutil.copy2(path, renders / f"{folder}__{path.name}")
if RENDER.is_file():
    shutil.copy2(RENDER, renders / RENDER.name)
print("saved:", len(list(renders.glob("*.wav"))), "harness renders,",
      len(list(Path("/kaggle/working/moss15_native").glob("*.wav"))) if native.is_dir() else 0, "native")
